# NB18: Long-Context Ablation

**Aim:** ModernBERT supports 8192 tokens via RoPE, but all prior experiments hardcode `max_length=512`. Earnings call transcripts (Source 8: median 161 words, max 2,596) are silently truncated. This notebook measures the truncation damage and tests whether longer context improves accuracy.

**Protocol:** Train ModernBERT+LoRA on aggregated data (FPB excluded) at max_length={512, 1024, 2048} × 3 seeds. Evaluate on FPB + aggregated test set with per-source breakdown.

**Runtime:** ~9 hours on T4 (9 training runs)

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers

In [ ]:
import json
import gc
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, training_args,
)
from peft import LoraConfig, get_peft_model, TaskType

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_DICT = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}
FPB_SOURCE = 5
MODEL_NAME = "answerdotai/ModernBERT-base"
SOURCE_NAMES = {3: "Earnings (narrative)", 4: "Press releases", 5: "FPB", 8: "Earnings (Q&A)", 9: "Tweets"}

CONTEXT_LENGTHS = [512, 1024, 2048]
SEEDS = [3407, 42, 123]

os.makedirs("../results", exist_ok=True)

# P100 GPU check — training requires a GPU with compute capability >= 7.0
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    gpu_name = torch.cuda.get_device_name()
    print(f"GPU: {gpu_name} (sm_{cap[0]}{cap[1]})")
    if cap[0] < 7:
        raise RuntimeError(
            f"{gpu_name} (compute capability {cap[0]}.{cap[1]}) is not supported by this "
            f"PyTorch build (needs sm_70+). Training requires a T4 or newer GPU. "
            f"Please re-run this notebook — Kaggle will assign a different GPU.")
else:
    raise RuntimeError("No GPU available. This notebook requires a GPU for training.")

## 1. Truncation Analysis

Tokenize all training samples WITHOUT truncation to measure true token lengths per source. This quantifies the signal lost at each `max_length`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ds_full = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds_full = ds_full.filter(lambda x: x["task"] == "sentiment")

stats_by_source = defaultdict(list)
for split in ["train", "validation", "test"]:
    for example in ds_full[split]:
        ids = tokenizer(example["text"], add_special_tokens=True)["input_ids"]
        stats_by_source[example["source"]].append(len(ids))

MAX_LENGTHS_REPORT = [128, 512, 1024, 2048]

header = f"{'Source':<25} {'N':>6} {'Med':>5} {'P95':>6} {'Max':>6}"
for ml in MAX_LENGTHS_REPORT:
    header += f" {'Trunc@'+str(ml):>10}"
print(header)
print("-" * len(header))

trunc_data = {}
for src_id in sorted(stats_by_source.keys()):
    lengths = stats_by_source[src_id]
    name = SOURCE_NAMES.get(src_id, f"Source {src_id}")
    row = f"{name:<25} {len(lengths):>6} {int(np.median(lengths)):>5} "
    row += f"{int(np.percentile(lengths, 95)):>6} {max(lengths):>6}"
    src_trunc = {}
    for ml in MAX_LENGTHS_REPORT:
        trunc = sum(1 for l in lengths if l > ml)
        row += f" {trunc/len(lengths):>9.1%}"
        src_trunc[ml] = round(trunc / len(lengths), 4)
    print(row)
    trunc_data[name] = {
        "n": len(lengths), "median": int(np.median(lengths)),
        "p95": int(np.percentile(lengths, 95)), "max": max(lengths),
        "truncation_rates": src_trunc,
    }

tokens_lost_src8 = stats_by_source.get(8, [])
if tokens_lost_src8:
    lost_at_512 = [l - 512 for l in tokens_lost_src8 if l > 512]
    print(f"\nSource 8 tokens lost at max_length=512: {len(lost_at_512)} samples, "
          f"mean={np.mean(lost_at_512):.0f} tokens, total={sum(lost_at_512):,} tokens")

with open("../results/truncation_analysis.json", "w") as f:
    json.dump(trunc_data, f, indent=2)
print("\nSaved to results/truncation_analysis.json")

## 2. Data Loading

In [ ]:
ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")

ds_no_fpb = ds.filter(lambda x: x["source"] != FPB_SOURCE)
remove_cols = [c for c in ds_no_fpb["train"].column_names if c not in ("text", "labels")]
ds_no_fpb = ds_no_fpb.map(
    lambda ex: {"text": ex["text"], "labels": np.eye(NUM_CLASSES)[LABEL_DICT[ex["label"]]].tolist()},
    remove_columns=remove_cols,
)

fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]

print(f"Train: {len(ds_no_fpb['train']):,}  |  Val: {len(ds_no_fpb['validation']):,}  |  Test: {len(ds_no_fpb['test']):,}")
print(f"FPB 50agree: {len(fpb_50):,}  |  FPB allAgree: {len(fpb_all):,}")

## 3. Training & Evaluation Functions

In [ ]:
def train_held_out(max_length, seed):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, torch_dtype=torch.float32, attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16, lora_alpha=32, target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
        lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    model = model.cuda()

    if max_length == 512 and seed == SEEDS[0]:
        model.print_trainable_parameters()

    def tok_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length)

    train_tok = ds_no_fpb["train"].map(tok_fn, batched=True)
    val_tok = ds_no_fpb["validation"].map(tok_fn, batched=True)

    if max_length <= 512:
        bs, ga = 8, 4
    elif max_length <= 1024:
        bs, ga = 4, 8
    else:
        bs, ga = 2, 16

    trainer = Trainer(
        model=model, processing_class=tokenizer,
        train_dataset=train_tok, eval_dataset=val_tok,
        args=TrainingArguments(
            output_dir=f"out_longctx_{max_length}_s{seed}",
            per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
            warmup_steps=10, fp16=True, bf16=False,
            optim=training_args.OptimizerNames.ADAMW_TORCH,
            learning_rate=2e-4, weight_decay=0.001, lr_scheduler_type="cosine",
            seed=seed, num_train_epochs=10,
            load_best_model_at_end=True, metric_for_best_model="eval_loss",
            greater_is_better=False, save_strategy="epoch", eval_strategy="epoch",
            logging_strategy="epoch", gradient_checkpointing=True, report_to="none",
        ),
        compute_metrics=lambda ep: {
            "accuracy": accuracy_score(ep[1].argmax(axis=-1), ep[0].argmax(axis=-1))
        },
    )
    trainer.train()
    model = model.cuda().eval()
    return model, tokenizer

In [ ]:
def run_inference(model, tokenizer, texts, max_length=512, batch_size=32):
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def evaluate_held_out(model, tokenizer, max_length):
    results = {}

    for name, fpb in [("fpb_50agree", fpb_50), ("fpb_allagree", fpb_all)]:
        preds = run_inference(model, tokenizer, fpb["sentence"], max_length)
        results[name] = {
            "accuracy": round(accuracy_score(fpb["label"], preds), 4),
            "macro_f1": round(f1_score(fpb["label"], preds, average="macro"), 4),
            "n": len(fpb),
        }

    test_split = ds["test"]
    test_texts = test_split["text"]
    test_labels = [LABEL_DICT[l] for l in test_split["label"]]
    test_sources = test_split["source"]
    test_preds = run_inference(model, tokenizer, test_texts, max_length)

    results["agg_test"] = {
        "accuracy": round(accuracy_score(test_labels, test_preds), 4),
        "macro_f1": round(f1_score(test_labels, test_preds, average="macro"), 4),
        "n": len(test_texts),
    }

    for src_id in sorted(set(test_sources)):
        mask = [i for i, s in enumerate(test_sources) if s == src_id]
        if len(mask) < 5:
            continue
        src_labels = [test_labels[i] for i in mask]
        src_preds = [test_preds[i] for i in mask]
        src_name = SOURCE_NAMES.get(src_id, f"source_{src_id}")
        results[f"src_{src_id}_{src_name}"] = {
            "accuracy": round(accuracy_score(src_labels, src_preds), 4),
            "macro_f1": round(f1_score(src_labels, src_preds, average="macro", zero_division=0), 4),
            "n": len(mask),
        }

    return results

## 4. Run Ablation: 3 context lengths × 3 seeds

In [ ]:
all_results = []

for ml in CONTEXT_LENGTHS:
    for seed in SEEDS:
        print(f"\n{'='*60}")
        print(f"max_length={ml}, seed={seed}")
        print(f"{'='*60}")

        model, tok = train_held_out(max_length=ml, seed=seed)
        results = evaluate_held_out(model, tok, max_length=ml)
        results["max_length"] = ml
        results["seed"] = seed
        all_results.append(results)

        print(f"  FPB 50agree: {results['fpb_50agree']['accuracy']:.4f}")
        print(f"  FPB allAgree: {results['fpb_allagree']['accuracy']:.4f}")
        print(f"  Agg test: {results['agg_test']['accuracy']:.4f}")
        for k, v in results.items():
            if k.startswith("src_") and isinstance(v, dict):
                print(f"  {k}: {v['accuracy']:.4f} (n={v['n']})")

        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

with open("../results/longctx_ablation.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\nAll {len(all_results)} runs complete. Saved to results/longctx_ablation.json")

## 5. Results Summary

In [ ]:
rows = []
for r in all_results:
    row = {
        "max_length": r["max_length"], "seed": r["seed"],
        "fpb_50_acc": r["fpb_50agree"]["accuracy"],
        "fpb_50_f1": r["fpb_50agree"]["macro_f1"],
        "fpb_all_acc": r["fpb_allagree"]["accuracy"],
        "fpb_all_f1": r["fpb_allagree"]["macro_f1"],
        "agg_test_acc": r["agg_test"]["accuracy"],
        "agg_test_f1": r["agg_test"]["macro_f1"],
    }
    for k, v in r.items():
        if k.startswith("src_") and isinstance(v, dict):
            row[f"{k}_acc"] = v["accuracy"]
    rows.append(row)

df = pd.DataFrame(rows)

print("=" * 90)
print("CONTEXT LENGTH ABLATION — per-run results")
print("=" * 90)
print(df.to_string(index=False, float_format="%.4f"))

summary = df.groupby("max_length").agg(["mean", "std"]).round(4)
print(f"\n{'='*90}")
print("SUMMARY (mean ± std over 3 seeds)")
print(f"{'='*90}")

for ml in CONTEXT_LENGTHS:
    sub = df[df["max_length"] == ml]
    print(f"\nmax_length = {ml}")
    for col in ["fpb_50_acc", "fpb_50_f1", "fpb_all_acc", "agg_test_acc"]:
        mean, std = sub[col].mean(), sub[col].std()
        print(f"  {col:20s}: {mean:.4f} ± {std:.4f}")
    src_cols = [c for c in sub.columns if c.startswith("src_") and c.endswith("_acc")]
    for col in src_cols:
        mean, std = sub[col].mean(), sub[col].std()
        print(f"  {col:20s}: {mean:.4f} ± {std:.4f}")

In [ ]:
from scipy import stats

acc_512 = df[df["max_length"] == 512]["fpb_50_acc"].values
acc_1024 = df[df["max_length"] == 1024]["fpb_50_acc"].values
acc_2048 = df[df["max_length"] == 2048]["fpb_50_acc"].values

t1, p1 = stats.ttest_rel(acc_512, acc_1024)
t2, p2 = stats.ttest_rel(acc_512, acc_2048)
t3, p3 = stats.ttest_rel(acc_1024, acc_2048)

print("Paired t-tests on FPB 50agree accuracy (3 paired seeds)")
print(f"  512 vs 1024: t={t1:.3f}, p={p1:.4f} {'*' if p1 < 0.05 else ''}")
print(f"  512 vs 2048: t={t2:.3f}, p={p2:.4f} {'*' if p2 < 0.05 else ''}")
print(f"  1024 vs 2048: t={t3:.3f}, p={p3:.4f} {'*' if p3 < 0.05 else ''}")

print(f"\nMean deltas (FPB 50agree acc):")
print(f"  1024 - 512:  {(acc_1024.mean() - acc_512.mean())*100:+.2f}pp")
print(f"  2048 - 512:  {(acc_2048.mean() - acc_512.mean())*100:+.2f}pp")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ("fpb_50_acc", "FPB 50agree Accuracy"),
    ("fpb_all_acc", "FPB allAgree Accuracy"),
    ("agg_test_acc", "Aggregated Test Accuracy"),
]

for ax, (col, title) in zip(axes, metrics):
    means = [df[df["max_length"] == ml][col].mean() for ml in CONTEXT_LENGTHS]
    stds = [df[df["max_length"] == ml][col].std() for ml in CONTEXT_LENGTHS]
    bars = ax.bar([str(ml) for ml in CONTEXT_LENGTHS], means, yerr=stds,
                  capsize=5, color=["#2196F3", "#FF9800", "#4CAF50"], edgecolor="white")
    ax.set_xlabel("max_length")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{m:.4f}", ha="center", fontsize=10)
    ax.set_ylim(min(means) - 0.05, max(means) + 0.03)

plt.suptitle("Context Length Ablation — ModernBERT+LoRA (3 seeds)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("longctx_ablation_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
src_cols = [c for c in df.columns if c.startswith("src_") and c.endswith("_acc")]
if src_cols:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(src_cols))
    width = 0.25
    colors = ["#2196F3", "#FF9800", "#4CAF50"]

    for i, ml in enumerate(CONTEXT_LENGTHS):
        sub = df[df["max_length"] == ml]
        means = [sub[c].mean() for c in src_cols]
        ax.bar(x + i * width, means, width, label=f"max_length={ml}", color=colors[i])

    labels = [c.replace("src_", "").replace("_acc", "") for c in src_cols]
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-Source Accuracy by Context Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("longctx_per_source.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
best_ml = df.groupby("max_length")["fpb_50_acc"].mean().idxmax()
best_acc = df.groupby("max_length")["fpb_50_acc"].mean().max()
baseline_acc = df[df["max_length"] == 512]["fpb_50_acc"].mean()

print(f"{'='*60}")
print(f"FINAL RESULT")
print(f"{'='*60}")
print(f"Best context length: {best_ml}")
print(f"FPB 50agree accuracy: {best_acc:.4f} (vs {baseline_acc:.4f} at 512)")
print(f"Delta: {(best_acc - baseline_acc)*100:+.2f}pp")